In [2]:
import os
import sys
import warnings
import copy
from contextlib import contextmanager

from tqdm import trange

import numpy as np
import pandas as pd
import scanpy as sc


import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
sns.set_context(
    "paper",
    rc={
        "axes.labelsize": 18,
        "axes.titlesize": 20,
        "legend.fontsize": 12,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
    }
)

import sys
sys.path.insert(1, '../.')
from McCauley_utils import all_data

sys.path.insert(1, '../../.')
from notebook_utils import clear_adata, get_split

sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
from scLEMBAS import latent_separation as ls

/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install

In [3]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'McCauley'

n_cores = 30
os.environ["OMP_NUM_THREADS"] = str(1)
os.environ["MKL_NUM_THREADS"] = str(1)
os.environ["OPENBLAS_NUM_THREADS"] = str(1)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(1)
os.environ["NUMEXPR_NUM_THREADS"] = str(1)

In [4]:
(sn_ppis, tf_adata, adata, expr, source_label, target_label, weight_label, 
 stimulation_label, inhibition_label, cat_col, pert_col, ctrl_pert) = all_data

In [18]:
@contextmanager
def suppress_all_output():
    """
    Suppress stdout, stderr, warnings, tqdm, and most parallel chatter.
    """
    # save originals
    old_stdout = sys.stdout
    old_stderr = sys.stderr

    try:
        # redirect stdout / stderr
        sys.stdout = open(os.devnull, 'w')
        sys.stderr = open(os.devnull, 'w')

        # silence warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            # silence tqdm globally
            os.environ["TQDM_DISABLE"] = "1"

            yield

    finally:
        # restore
        sys.stdout.close()
        sys.stderr.close()
        sys.stdout = old_stdout
        sys.stderr = old_stderr

        os.environ.pop("TQDM_DISABLE", None)

# as in Notebook 04
csw = {
    'max_components': 25 ,
    'metric': 'accuracy', 
    'method': 'elbow', 
    'n_folds': 5, 
    'seed': 888
}

assessment_kwargs = {
    'n_perm': 100, 
    'get_q2_pval': True, 
    'get_r2_pval': False, 
    'get_accuracy_pval': False,
    'n_folds': 5, 
    'seed': 888
}
        
def run_pls_(tf_adata_sub, n_components = None):
    with suppress_all_output():
        models, X_pls = ls.pls_da(
            adata = tf_adata_sub, 
            n_components = n_components, 
            assess = True, 
            enc_X = None, enc_Y = None, control_confounders = None, 
            separate_by = 'both', 
            pert_col = pert_col, 
            cat_col = 'secretory_mcc_subsets', 
            component_selection_kwargs = csw, 
            assessment_kwargs = assessment_kwargs, 
            n_cores = n_cores, 
            verbose = False
        )
        

In [15]:
has_subtype = ['Club', 'Goblet', 'Multiciliated']

ct_pert_map = {}
for ct in tf_adata.obs[cat_col].cat.categories:
    if ct not in has_subtype:
        continue
    ct_pert_map[ct] = sorted(tf_adata.obs[tf_adata.obs[cat_col] == ct][pert_col].unique())
    ct_pert_map[ct].remove(ctrl_pert)

In [16]:
for ct, perts in ct_pert_map.items():
    for pert in perts:
        ct_mask = tf_adata.obs[cat_col] == ct
        pert_mask = tf_adata.obs[pert_col].isin([ctrl_pert, pert])
        
        tf_adata_sub = tf_adata[ct_mask & pert_mask].copy()
        tf_adata_sub = clear_adata(tf_adata_sub)
        
        break
    break

PLS Assessment Permutations:   0%|                      | 0/100 [01:15<?, ?it/s]


In [19]:
with suppress_all_output():
    models, X_pls = ls.pls_da(
        adata = tf_adata_sub, 
        n_components = n_components, 
        assess = True, 
        enc_X = None, enc_Y = None, control_confounders = None, 
        separate_by = 'both', 
        pert_col = pert_col, 
        cat_col = 'secretory_mcc_subsets', 
        component_selection_kwargs = csw, 
        assessment_kwargs = assessment_kwargs, 
        n_cores = n_cores, 
        verbose = False
    )